# Task-token allocation: run on Colab
Runtime -> Change runtime type -> GPU (A100 or L4 if you have Colab Pro, else T4).
`data/` and `runs/` live on Google Drive, so a disconnect loses nothing; finished seeds are kept.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/VIP'
!mkdir -p $DRIVE/data $DRIVE/runs

In [ ]:
# Clone the repo (private repo: use a GitHub token, or upload the folder to Drive instead)
!git clone -b claude/kind-carson-96re7c https://github.com/karlkhoury/vip.git /content/VIP
%cd /content/VIP
!pip -q install -r requirements.txt
import os
# data/ and every runs/<name>/ folder live on Drive (re-run this cell after a restart)
if not os.path.islink('data'):
    os.symlink(f'{DRIVE}/data', 'data')
for name in os.listdir(f'{DRIVE}/runs'):
    if not os.path.exists(f'runs/{name}'):
        os.symlink(f'{DRIVE}/runs/{name}', f'runs/{name}')


In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!python -m tests.test_all

In [ ]:
# Build the dataset once (skipped if it already exists on Drive)
import os
if not os.path.exists('data/phrasebank_multitask.parquet'):
    !python -m common.prepare_data --out data/phrasebank_multitask.parquet

In [ ]:
# Run experiments one config at a time. Outputs go to Drive.
def run(cfg, seeds=None):
    name = cfg.replace('.yaml', '')
    os.makedirs(f'{DRIVE}/runs/{name}', exist_ok=True)
    if not os.path.islink(f'runs/{name}'):
        os.symlink(f'{DRIVE}/runs/{name}', f'runs/{name}')
    s = '' if seeds is None else '--seeds ' + ' '.join(map(str, seeds))
    !python run.py --config runs/configs/{cfg} {s}

run('exp1_saturation.yaml')
run('exp2_main_cap8.yaml')

In [ ]:
run('legacy_B1_32sym.yaml')
run('legacy_B3_32sym.yaml')
run('exp3_priority.yaml')
for a in ['sentence', 'snr', 'channel']:
    run(f'exp4_no_{a}.yaml')

In [ ]:
!python -m common.analyze runs/exp2_main_cap8 runs/legacy_B1_32sym runs/legacy_B3_32sym --ref allocator --out $DRIVE/report_cap8